In [108]:
# Installing all the required libraries

%pip install -q pandas numpy scikit-learn==1.6.1 matplotlib seaborn joblib streamlit openpyxl

Note: you may need to restart the kernel to use updated packages.


In [110]:
# Importing all the required libraries

import pandas as pd
import numpy as np

print("All libraries imported successfully!")

All libraries imported successfully!


In [75]:
# Loading the UCI Dry Bean dataset from the uploaded Excel file

df = pd.read_excel("Dry_Bean_Dataset.xlsx", sheet_name="Dry_Beans_Dataset")

X = df.drop(columns=["Class"]).copy()
y = df["Class"].copy()

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

Dataset loaded successfully!
Dataset shape: (13611, 17)


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [76]:

# Verifying the required dataset criteria

print("Total records:", df.shape[0])
print("Total input features:", df.shape[1] - 1)

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nClass distribution:")
print(df["Class"].value_counts())

# Assignment dataset requirements:-
assert df.shape[0] >= 500, "Not enough records!"
assert df.shape[1] - 1 >= 12, "Not enough features!"
assert "Class" in df.columns, "Target column missing!"

print("\nSUCCESS: Dataset satisfies the minimum size and feature requirements!")


Total records: 13611
Total input features: 16

Column names:
['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4', 'Class']

Missing values:
0

Class distribution:
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64

SUCCESS: Dataset satisfies the minimum size and feature requirements!


In [77]:

# Preparing the training and testing data

from sklearn.model_selection import train_test_split

# Separate out input features and target variable

X = df.drop(columns=["Class"])
y = df["Class"].astype(str)

# Here, Splitting the data into 80% training and 20% testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Data preparation completed successfully!")
print("---------------------------------------")
print("Total records:", len(df))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Number of input features:", X_train.shape[1])
print("Number of classes:", y.nunique())
print("\nClass labels:")
print(sorted(y.unique()))


Data preparation completed successfully!
---------------------------------------
Total records: 13611
Training records: 10888
Testing records: 2723
Number of input features: 16
Number of classes: 7

Class labels:
['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA']


In [78]:

# Training out and evaluating all required five ML models

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

import pandas as pd

# Defining  all five machine learning models

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, random_state=42))
    ]),

    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5, n_jobs=-1))
    ]),

    "Gaussian Naive Bayes": GaussianNB(),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="linear", probability=True, random_state=42))
    ])
}

# Storing evaluation results

results = []
trained_models = {}

# Train and evaluate each model

for model_name, model in models.items():

    print("\nTraining:", model_name)

    # Train model
    model.fit(X_train, y_train)

    # Making predictions
    y_pred = model.predict(X_test)

    # Calculate class probabilities for AUC

    y_probability = model.predict_proba(X_test)

    # Calculate evaluation metrics

    accuracy = accuracy_score(y_test, y_pred)

    auc = roc_auc_score(
        y_test,
        y_probability,
        multi_class="ovr",
        average="macro",
        labels=model.classes_
    )

    precision = precision_score(
        y_test, y_pred, average="macro", zero_division=0
    )

    recall = recall_score(
        y_test, y_pred, average="macro", zero_division=0
    )

    f1 = f1_score(
        y_test, y_pred, average="macro", zero_division=0
    )

    mcc = matthews_corrcoef(y_test, y_pred)

    # Store results

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "AUC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "MCC": mcc
    })

    trained_models[model_name] = model

    print("Completed:", model_name)

# Create comparison table

results_df = pd.DataFrame(results)

print("\n======================================")
print("FINAL MODEL PERFORMANCE COMPARISON")
print("======================================")

display(results_df.round(4))
print("\nAll five models trained and evaluated successfully!")



Training: Logistic Regression


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Completed: Logistic Regression

Training: Decision Tree
Completed: Decision Tree

Training: KNN
Completed: KNN

Training: Gaussian Naive Bayes
Completed: Gaussian Naive Bayes

Training: SVM
Completed: SVM

FINAL MODEL PERFORMANCE COMPARISON


,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.9214,0.9948,0.9354,0.9321,0.9335,0.9050
1,Decision Tree,0.8920,0.9450,0.9075,0.9090,0.9081,0.8696
2,KNN,0.9166,0.9833,0.9320,0.9271,0.9293,0.8992
3,Gaussian Naive Bayes,0.7639,0.9672,0.7744,0.7694,0.7677,0.7154
4,SVM,0.9221,0.9951,0.9353,0.9320,0.9335,0.9059



All five models trained and evaluated successfully!


In [79]:

# Saving  model comparison results

results_df.to_csv("model_comparison_results.csv", index=False)

print("Model comparison results saved successfully!")

print("\nBest model based on Accuracy:")
best_model = results_df.loc[results_df["Accuracy"].idxmax()]
print(best_model)


Model comparison results saved successfully!

Best model based on Accuracy:
Model             SVM
Accuracy     0.922145
AUC          0.995122
Precision    0.935314
Recall       0.931966
F1 Score     0.933536
MCC          0.905854
Name: 4, dtype: object


In [80]:

# Saving all five trained models

import os
import joblib

# Create model folder

os.makedirs("model", exist_ok=True)

# Save each trained model

model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Gaussian Naive Bayes": "gaussian_nb.joblib",
    "SVM": "svm.joblib"
}

for model_name, filename in model_files.items():

    joblib.dump(
        trained_models[model_name],
        os.path.join("model", filename)
    )

    print("Saved:", filename)

print("\nAll five models saved successfully!")


Saved: logistic_regression.joblib
Saved: decision_tree.joblib
Saved: knn.joblib
Saved: gaussian_nb.joblib
Saved: svm.joblib

All five models saved successfully!


In [81]:

# Creating test dataset CSV

test_data = X_test.copy()

# Add actual target values
test_data["Class"] = y_test.values

# Saving test data

test_data.to_csv("test_data.csv", index=False)

print("Test dataset created successfully!")
print("Test dataset shape:", test_data.shape)

print("\nFirst five records:")
display(test_data.head())

print("\nTarget column included:", "Class" in test_data.columns)


Test dataset created successfully!
Test dataset shape: (2723, 17)

First five records:


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
12803,36002,700.889,257.888986,178.268649,1.446631,0.722605,36368,214.100841,0.797386,0.989936,0.920954,0.830205,0.007163,0.002099,0.689241,0.997078,DERMASON
3713,183825,1594.423,582.225240,402.762055,1.445581,0.722124,185116,483.790512,0.797852,0.993026,0.908672,0.830934,0.003167,0.000931,0.690451,0.998102,BOMBAY
8974,45595,801.717,301.092828,193.505323,1.555992,0.766138,46054,240.942643,0.769926,0.990033,0.891425,0.800227,0.006604,0.001670,0.640363,0.996401,SIRA
1743,45084,763.811,261.138204,219.977872,1.187111,0.538882,45425,239.588672,0.773098,0.992493,0.971092,0.917478,0.005792,0.002532,0.841767,0.999271,SEKER
2303,61645,952.199,335.139136,235.843970,1.421021,0.710478,62499,280.158619,0.795522,0.986336,0.854382,0.835947,0.005437,0.001638,0.698808,0.993020,BARBUNYA



Target column included: True


In [82]:

%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import os

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

# 1. Streamlit page configuration

st.set_page_config(
    page_title="Dry Bean Classification",
    page_icon="🌱",
    layout="wide"
)

st.title("🌱 Dry Bean Classification")
st.write(
    "Machine Learning Model Comparison and Evaluation "
    "using the UCI Dry Bean Dataset."
)

st.markdown("---")

# 2. Model file configuration

MODEL_FILES = {
    "Logistic Regression": "model/logistic_regression.joblib",
    "Decision Tree": "model/decision_tree.joblib",
    "KNN": "model/knn.joblib",
    "Gaussian Naive Bayes": "model/gaussian_nb.joblib",
    "SVM": "model/svm.joblib"
}

# 3. Load saved models

@st.cache_resource
def load_models():

    loaded_models = {}

    for model_name, model_path in MODEL_FILES.items():

        if not os.path.exists(model_path):
            raise FileNotFoundError(
                f"Model file not found: {model_path}"
            )

        loaded_models[model_name] = joblib.load(model_path)

    return loaded_models


try:
    trained_models = load_models()
    st.success("All five trained models loaded successfully!")

except Exception as e:
    st.error(f"Unable to load models: {e}")
    st.stop()

# 4. Upload test dataset

st.header("1. Upload Test Dataset")

st.write(
    "Upload a CSV file containing all 16 input features "
    "and the actual Class column."
)

uploaded_file = st.file_uploader(
    "Choose your test CSV file",
    type=["csv"]
)


# 5. Select machine learning model

st.header("2. Select Machine Learning Model")

selected_model_name = st.selectbox(
    "Choose a model:",
    list(MODEL_FILES.keys())
)

selected_model = trained_models[selected_model_name]

st.info(f"Selected model: {selected_model_name}")

# 6. Evaluate uploaded dataset

if uploaded_file is not None:

    try:

        test_df = pd.read_csv(uploaded_file)

        st.header("3. Dataset Information")

        st.write("Uploaded dataset shape:", test_df.shape)

        st.dataframe(test_df.head())

        # Check target column
        if "Class" not in test_df.columns:
            st.error(
                "The uploaded CSV must contain the target column 'Class'."
            )
            st.stop()

        # Get input feature names from trained model
        required_features = list(selected_model.feature_names_in_)

        missing_features = [
            col for col in required_features
            if col not in test_df.columns
        ]

        if missing_features:
            st.error(
                f"Missing required input features: {missing_features}"
            )
            st.stop()
        # Prepare input features and target values
        X_test_upload = test_df[required_features].copy()

        if test_df["Class"].isnull().any():
            st.error("The target column 'Class' contains missing values.")
            st.stop()

        y_test_upload = test_df["Class"].astype(str)

        # Validate numeric input features
        try:
            X_test_upload = X_test_upload.apply(
            pd.to_numeric, errors="raise"
            )
        except (ValueError, TypeError):
            st.error("All input features must contain numeric values.")
            st.stop()

        # Check for missing input values
        if X_test_upload.isnull().any().any():
            st.error("The input features contain missing values.")
            st.stop()

        # Make predictions using the saved model
        y_pred = selected_model.predict(X_test_upload)

        # Get predicted probabilities for multiclass AUC
        y_probability = selected_model.predict_proba(
            X_test_upload
        )

        classes = list(selected_model.classes_)

        # Calculate evaluation metrics
        accuracy = accuracy_score(y_test_upload, y_pred)

        precision = precision_score(
            y_test_upload,
            y_pred,
            average="macro",
            zero_division=0
        )

        recall = recall_score(
            y_test_upload,
            y_pred,
            average="macro",
            zero_division=0
        )

        f1 = f1_score(
            y_test_upload,
            y_pred,
            average="macro",
            zero_division=0
        )

        mcc = matthews_corrcoef(y_test_upload, y_pred)

        try:
            auc = roc_auc_score(
                y_test_upload,
                y_probability,
                labels=classes,
                multi_class="ovr",
                average="macro"
            )
            auc_display = f"{auc:.4f}"

        except ValueError:
            auc_display = "Not available for this test sample"

        # 7. Display evaluation metrics

        st.markdown("---")
        st.header("4. Model Evaluation Results")

        col1, col2, col3 = st.columns(3)

        col1.metric("Accuracy", f"{accuracy:.4f}")
        col2.metric("AUC Score", auc_display)
        col3.metric("Precision", f"{precision:.4f}")

        col4, col5, col6 = st.columns(3)

        col4.metric("Recall", f"{recall:.4f}")
        col5.metric("F1 Score", f"{f1:.4f}")
        col6.metric("MCC", f"{mcc:.4f}")

        # 8. Display confusion matrix

        st.markdown("---")
        st.header("5. Confusion Matrix")

        cm = confusion_matrix(
            y_test_upload,
            y_pred,
            labels=classes
        )

        fig, ax = plt.subplots(figsize=(10, 7))

        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=classes,
            yticklabels=classes,
            ax=ax
        )

        ax.set_xlabel("Predicted Class")
        ax.set_ylabel("Actual Class")
        ax.set_title(f"Confusion Matrix - {selected_model_name}")

        plt.xticks(rotation=45)
        plt.yticks(rotation=0)
        plt.tight_layout()

        st.pyplot(fig)
        plt.close(fig)

        # 9. Classification report
        # ---------------------------------------

        st.markdown("---")
        st.header("6. Classification Report")

        report = classification_report(
            y_test_upload,
            y_pred,
            labels=classes,
            target_names=classes,
            zero_division=0,
            output_dict=True
        )

        report_df = pd.DataFrame(report).transpose()

        st.dataframe(
            report_df.style.format("{:.4f}"),
            use_container_width=True
        )

        # ---------------------------------------
        # 10. Completion message
        # ---------------------------------------

        st.success(
            "Evaluation completed successfully! "
            "The saved model was used without retraining."
        )

    except Exception as e:

        st.error(f"An error occurred during evaluation: {e}")

else:

    st.info(
        "Please upload test_data.csv to view model evaluation results."
    )


Overwriting app.py


In [83]:
%%writefile requirements.txt

streamlit
pandas
numpy
scikit-learn==1.6.1
matplotlib
seaborn
joblib

Overwriting requirements.txt


In [84]:
%%writefile README.md

# Dry Bean Classification Using Machine Learning

## 1. Problem Statement

The objective of this project is to classify dry bean varieties
using machine learning classification algorithms based on
physical measurements of dry bean samples.

This is a multiclass classification problem involving seven
different dry bean varieties.

## 2. Dataset Description

Dataset: UCI Dry Bean Dataset

Source: https://archive.ics.uci.edu/dataset/602/dry

Total Instances: 13,611

Input Features: 16

Target Variable: Class

Number of Classes: 7

The dataset contains numerical measurements of dry bean samples.
The target variable represents the corresponding bean variety.

## 3. Data Preparation

- Checked dataset dimensions and missing values.
- Separated input features and target variable.
- Used an 80:20 training and testing split.
- Applied stratified sampling to preserve class distribution.
- Used StandardScaler within pipelines for Logistic Regression,
  KNN and SVM.

## 4. Machine Learning Models

Five classification algorithms were implemented:

1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbors (KNN)
4. Gaussian Naive Bayes
5. Support Vector Machine (SVM)

## 5. Evaluation Metrics

The following metrics were calculated:

- Accuracy
- AUC Score
- Precision
- Recall
- F1 Score
- Matthews Correlation Coefficient (MCC)

Precision, Recall and F1 Score use macro averaging.
AUC is calculated using the One-vs-Rest approach with macro averaging.

## 6. Model Performance Comparison

| Model | Accuracy | AUC | Precision | Recall | F1 Score | MCC |
|---|---:|---:|---:|---:|---:|---:|
| Logistic Regression | 0.9214 | 0.9948 | 0.9354 | 0.9321 | 0.9335 | 0.9050 |
| Decision Tree | 0.8920 | 0.9450 | 0.9075 | 0.9090 | 0.9081 | 0.8696 |
| KNN | 0.9166 | 0.9833 | 0.9320 | 0.9271 | 0.9293 | 0.8992 |
| Gaussian Naive Bayes | 0.7639 | 0.9672 | 0.7744 | 0.7694 | 0.7677 | 0.7154 |
| SVM | 0.9221 | 0.9951 | 0.9353 | 0.9320 | 0.9335 | 0.9059 |

## 7. Observations

### Logistic Regression

Logistic Regression achieved an accuracy of 92.14%.
It obtained a high AUC score of 0.9948.
Its Precision and Recall were slightly higher than SVM
in this experiment.

### Decision Tree Classifier

The Decision Tree achieved an accuracy of 89.20%.
Its performance was lower than Logistic Regression and KNN
in this experiment.

### K-Nearest Neighbors

KNN achieved an accuracy of 91.66%.
Its performance was close to Logistic Regression and SVM.
Feature scaling was applied before model training.

### Gaussian Naive Bayes

Gaussian Naive Bayes achieved an accuracy of 76.39%.
Although its AUC score was 0.9672, its accuracy and other
classification metrics were lower than those of the other models.

### Support Vector Machine

SVM achieved the highest accuracy of 92.21% and the highest
AUC score of 0.9951.
It also achieved the highest MCC of 0.9059.
Its F1 Score was equal to Logistic Regression at four decimal places.

## 8. Overall Model Comparison and Conclusion

Based on the evaluation results, Support Vector Machine
performed best overall among the five implemented models.

Logistic Regression also showed strong performance,
with slightly higher Precision and Recall than SVM.
KNN achieved competitive results.

Gaussian Naive Bayes had comparatively lower accuracy,
while the Decision Tree achieved moderate performance.

The experiment demonstrates that different classification
algorithms can produce different results on the same dataset.

## 9. Streamlit Application

The Streamlit application provides:

- CSV test dataset upload.
- Selection of one of the five trained models.
- Display of Accuracy, AUC, Precision, Recall, F1 Score and MCC.
- Confusion matrix.
- Classification report.

The application loads the saved trained models and evaluates
the uploaded test dataset without retraining the models.

## 10. Project Files

- app.py
- requirements.txt
- README.md
- test_data.csv
- model_comparison_results.csv
- model/
  - logistic_regression.joblib
  - decision_tree.joblib
  - knn.joblib
  - gaussian_nb.joblib
  - svm.joblib

## 11. GitHub Repository

https://github.com/anix4632/dry-bean-ml-classification

## 12. Dataset Reference

UCI Machine Learning Repository:
Dry Bean Dataset

DOI: https://doi.org/10.24432/C50S4B

Overwriting README.md


In [85]:

# Verify complete project files

import os

required_files = [
    "app.py",
    "requirements.txt",
    "README.md",
    "test_data.csv",
    "model_comparison_results.csv",
    "model/logistic_regression.joblib",
    "model/decision_tree.joblib",
    "model/knn.joblib",
    "model/gaussian_nb.joblib",
    "model/svm.joblib"
]

print("PROJECT FILE CHECK")
print("==================")

all_files_present = True

for file in required_files:

    if os.path.exists(file):
        print("PASS:", file)
    else:
        print("MISSING:", file)
        all_files_present = False

if all_files_present:
    print("\nSUCCESS: All required project files are present!")
else:
    print("\nWARNING: Some project files are missing.")


PROJECT FILE CHECK
PASS: app.py
PASS: requirements.txt
PASS: README.md
PASS: test_data.csv
PASS: model_comparison_results.csv
PASS: model/logistic_regression.joblib
PASS: model/decision_tree.joblib
PASS: model/knn.joblib
PASS: model/gaussian_nb.joblib
PASS: model/svm.joblib

SUCCESS: All required project files are present!


In [86]:

# Check app.py for syntax errors

import py_compile
import os

assert os.path.exists("app.py"), "app.py is missing!"

py_compile.compile("app.py", doraise=True)

print("SUCCESS: app.py has no Python syntax errors!")


SUCCESS: app.py has no Python syntax errors!


In [87]:

# Test all saved models using test_data.csv

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score

# Load test dataset
test_df = pd.read_csv("test_data.csv")

X_check = test_df.drop(columns=["Class"])
y_check = test_df["Class"].astype(str)

print("TESTING SAVED MODELS")
print("====================")

for model_name, filename in model_files.items():

    # Load saved model
    saved_model = joblib.load(os.path.join("model", filename))

    # Generate predictions
    predictions = saved_model.predict(X_check)

    # Calculate accuracy
    accuracy = accuracy_score(y_check, predictions)

    # Calculate AUC
    probabilities = saved_model.predict_proba(X_check)

    auc = roc_auc_score(
        y_check,
        probabilities,
        labels=saved_model.classes_,
        multi_class="ovr",
        average="macro"
    )

    print("\nModel:", model_name)
    print("Accuracy:", round(accuracy, 4))
    print("AUC:", round(auc, 4))
    print("PASS: Saved model loaded and predicted successfully.")

print("\nALL FIVE SAVED MODELS PASSED THE TEST!")


TESTING SAVED MODELS

Model: Logistic Regression
Accuracy: 0.9214
AUC: 0.9948
PASS: Saved model loaded and predicted successfully.

Model: Decision Tree
Accuracy: 0.892
AUC: 0.945
PASS: Saved model loaded and predicted successfully.

Model: KNN
Accuracy: 0.9166
AUC: 0.9833
PASS: Saved model loaded and predicted successfully.

Model: Gaussian Naive Bayes
Accuracy: 0.7639
AUC: 0.9672
PASS: Saved model loaded and predicted successfully.

Model: SVM
Accuracy: 0.9221
AUC: 0.9951
PASS: Saved model loaded and predicted successfully.

ALL FIVE SAVED MODELS PASSED THE TEST!


In [88]:

# Create the final project folder and ZIP file

import os
import shutil

project_folder = "Dry_Bean_Classification_Project"

# Remove an earlier package, if present
if os.path.exists(project_folder):
    shutil.rmtree(project_folder)

os.makedirs(project_folder)

# Copy the required project files
project_files = [
    "app.py",
    "requirements.txt",
    "README.md",
    "test_data.csv",
    "model_comparison_results.csv"
]

for filename in project_files:
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Missing file: {filename}")
    shutil.copy2(filename, project_folder)

# Copy all five saved models
if not os.path.isdir("model"):
    raise FileNotFoundError("Model folder is missing")

shutil.copytree(
    "model",
    os.path.join(project_folder, "model")
)

# Create the ZIP package
zip_path = shutil.make_archive(
    project_folder,
    "zip",
    root_dir=".",
    base_dir=project_folder
)

# Confirm successful creation
assert os.path.exists(zip_path)

print("Final project package created successfully!")
print("ZIP file:", zip_path)
print("ZIP size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")

print("\nFiles included:")
for root, folders, filenames in os.walk(project_folder):
    for filename in filenames:
        print(os.path.join(root, filename))


Final project package created successfully!
ZIP file: /home/s2026aj05204/Dry_Bean_Classification_Project.zip
ZIP size: 1.91 MB

Files included:
Dry_Bean_Classification_Project/app.py
Dry_Bean_Classification_Project/requirements.txt
Dry_Bean_Classification_Project/README.md
Dry_Bean_Classification_Project/test_data.csv
Dry_Bean_Classification_Project/model_comparison_results.csv
Dry_Bean_Classification_Project/model/logistic_regression.joblib
Dry_Bean_Classification_Project/model/decision_tree.joblib
Dry_Bean_Classification_Project/model/knn.joblib
Dry_Bean_Classification_Project/model/gaussian_nb.joblib
Dry_Bean_Classification_Project/model/svm.joblib


In [89]:
import os
import zipfile

zip_file = "Dry_Bean_Classification_Project.zip"
project_folder = "Dry_Bean_Classification_Project"

with zipfile.ZipFile(zip_file, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk(project_folder):
        for file in files_list:
            file_path = os.path.join(root, file)
            archive_path = os.path.relpath(file_path, ".")
            zipf.write(file_path, archive_path)

print("ZIP created successfully!")
print("ZIP file:", os.path.abspath(zip_file))
print("ZIP size:", round(os.path.getsize(zip_file) / (1024 * 1024), 2), "MB")

ZIP created successfully!
ZIP file: /home/s2026aj05204/Dry_Bean_Classification_Project.zip
ZIP size: 1.91 MB
